# Figure 5 · Reconstruction accuracy and cost

**Question:** What does each model gain over interpolation, and at what inference cost?

**Status:** experiment inventory is available; the matched physical-error and
timing benchmark is pending. The inventory is not an accuracy ranking. Historical
unverified figures and tiny debugging pilots are excluded from the main comparison.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Works from the repository root or from paper_figures/.
here = Path.cwd()
workspace = here if (here / 'figure_utils.py').exists() else here / 'paper_figures'
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))
from figure_utils import (ROOT, FIGURES, DATA, COLORS, style, read_json, export,
                          panel, box, arrow, verified_sources, completed_evaluation,
                          run_inventory, sha)
%matplotlib inline
style()
VERSION = 'v01'

In [2]:
inventory = run_inventory()
display(inventory.drop(columns=['wandb']))
display(Markdown('Checkpoint selection differs: residuals use full-validation masked MAE; '
    'diffusion uses generated physical MAE on a fixed 256-object validation monitor. '
    'Equal update budgets do not match loss, representation, capacity or inference cost.'))

,model,state,epoch,planned_epochs,batch,parameters,validation_monitor,representation,selection
0,Residual legacy,complete,50,50,8,2332177,2118,linear,masked normalized MAE
1,Residual direct 5×,complete,50,50,8,2959953,2118,linear,masked normalized MAE
2,Corrected diffusion,running / inspect status,4,50,8,4910593,256,asinh,generated physical MAE


Checkpoint selection differs: residuals use full-validation masked MAE; diffusion uses generated physical MAE on a fixed 256-object validation monitor. Equal update budgets do not match loss, representation, capacity or inference cost.

In [3]:
benchmark_path = FIGURES / 'inputs/model_benchmark.json'
ready = benchmark_path.exists()
if not ready:
    display(Markdown('**Waiting for a completed, same-object benchmark in physical units.** '
                     'Training-loss curves are deliberately not plotted as a model ranking.'))
else:
    benchmark = read_json(benchmark_path)
    assert benchmark['complete'] and benchmark['split'] == 'validation' and benchmark['units'] == 'MJy/sr'
    object_ids = benchmark['object_ids']
    assert len(set(object_ids)) == len(object_ids) and object_ids
    assert benchmark['timing_protocol'] and benchmark['population_rule']
    rows = []
    for name, values in benchmark['methods'].items():
        assert values['object_ids'] == object_ids
        mae, rmse = np.asarray(values['mae_mjy_sr']), np.asarray(values['rmse_mjy_sr'])
        assert mae.shape == rmse.shape == (len(object_ids),)
        assert np.isfinite(mae).all() and np.isfinite(rmse).all()
        assert (mae >= 0).all() and (rmse >= 0).all()
        assert np.isfinite(values['inference_ms_per_object']) and values['inference_ms_per_object'] > 0
        rows.append(dict(model=name, mae=mae.mean(), rmse=rmse.mean(),
                         milliseconds=values['inference_ms_per_object']))
    metrics = pd.DataFrame(rows)
    display(metrics)

**Waiting for a completed, same-object benchmark in physical units.** Training-loss curves are deliberately not plotted as a model ranking.

In [4]:
if ready:
    fig, axes = plt.subplots(1, 3, figsize=(8.5, 3.2), layout='constrained')
    for ax, key, title, unit in zip(axes, ['mae', 'rmse', 'milliseconds'],
        ['Mean absolute error', 'Mean per-object pixel RMSE', 'Inference cost'],
        ['MJy/sr', 'MJy/sr', 'ms / object']):
        ax.barh(metrics.model, metrics[key], color=COLORS['legacy'])
        ax.invert_yaxis(); ax.set_xlabel(unit); ax.set_title(title)
    fig.suptitle(f'Same {len(object_ids):,} validation galaxies · matched valid pixels', fontsize=10)
    plt.show()
    paths = export(fig, f'fig05_model_comparison_{VERSION}', '05_model_comparison.ipynb',
        [benchmark_path], dict(objects=len(object_ids), timing=benchmark['timing_protocol']),
        status='matched validation reconstruction benchmark draft')

## Working caption and review choices

Physical-intensity reconstruction errors on identical validation galaxies and
valid target pixels, together with measured inference cost under a fixed timing
protocol. Diffusion sampling steps and ensemble size must appear in the caption.

- Use native/bilinear/bicubic controls where a target-grid comparison is defined.
- Keep paired block uncertainty in the final benchmark; the initial layout
  has no invented error bars and is not publication-ready without them.
- PSNR/SSIM need an explicit physical range/window contract before inclusion.
- Separate quality from efficiency; do not infer a model winner from equal epochs alone.
- Galaxy shapes are the primary scientific result and receive their own figure.